VAE viewer notebook for JetBot
===

This notebook can visualize reconstructioned image by vae. This repository using JetBot camera.

Updated with debugging tools, undistortion, center crop, color channel fixes (BGR to RGB), and range adjustments for better compatibility with training.
Fixed VGG import for older torchvision versions (using pretrained=True instead of weights).
Fixed color panel dtype to uint8 for cv2 compatibility.

In [ ]:
import sys
import PIL
import numpy as np
import cv2
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import vgg16
from jetbot import Camera, bgr8_to_jpeg
from learning_racer.vae import VAE

## Setting Parameter

|Name | Description| Default|
|:----|:-----------|:-------|
|IMAGE_CHANNELS | Image channel such as RGB | 3 Not change|
|VARIANTS_SIZE  | Variants size of VAE      | 32          |
|MODEL_PATH     | Trained VAE model file path | ../../vae.torch|

In [ ]:
IMAGE_CHANNELS = 3
VARIANTS_SIZE = 128
MODEL_PATH = '../../../vae_improved5.torch'

## Load trained VAE model.
Loading trained VAE model on GPU memory. 

In [ ]:
device = torch.device('cuda')
vae = VAE(image_channels=IMAGE_CHANNELS, z_dim=VARIANTS_SIZE)
vae.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(device)), strict=False)
vae.to(device).eval()

## Load VGG for Perceptual Loss

In [ ]:
vgg = vgg16(pretrained=True).features[:16].eval().to(device)
for param in vgg.parameters():
    param.requires_grad = False

vgg_normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

## Create camera 


In [ ]:
camera = Camera.instance(width=320, height=240)

## Define preprocess and postprocess

Includes fisheye undistortion with scaled K/D for 320x240 resolution (original calibration for higher res, scaled by 0.25).
Added BGR to RGB conversion for correct colors.

In [ ]:
# Scaled K and D for 320x240 (original for ~1280x960, scale factor 0.25)
scale = 0.25
K = np.array([
    [600.79112202 * scale, 0, 626.87737459 * scale],
    [0, 598.85259306 * scale, 454.59767681 * scale],
    [0, 0, 1]
], dtype=np.float32)
D = np.array([[-0.0043877, -0.0456547, 0.05200435, -0.0220991]], dtype=np.float32)

preprocess_transform = transforms.Compose([
    transforms.Resize((120, 160)),  # (height, width)
    transforms.CenterCrop((80, 160)),
    transforms.ToTensor()  # [0,1]
])

def preprocess(image_np):
    # Undistort the BGR image
    undistorted = cv2.fisheye.undistortImage(image_np, K, D, None, K)
    # Convert BGR to RGB
    undistorted_rgb = cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB)
    # Convert to PIL
    observe = PIL.Image.fromarray(undistorted_rgb)
    # Apply transforms
    tensor = preprocess_transform(observe)
    return tensor

def np_to_jpeg(image_np):
    # Assumes image_np is RGB; convert to BGR for cv2.imencode
    if image_np.shape[2] == 3:  # Ensure 3 channels
        image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    else:
        image_bgr = image_np
    return bytes(cv2.imencode('.jpg', image_bgr)[1])

## Visualize latent space function

In [ ]:
ABS_LATENT_MAX_VALUE = 3
PANEL_HEIGHT = 10
PANEL_WIDTH = 10

def sigmoid(x, gain=1, offset_x=0):
    return ((np.tanh(((x+offset_x)*gain)/2)+1)/2)

def color_bar_rgb(x):
    gain = 10
    offset_x= 0.2
    offset_green = 0.6
    x = (x * 2) - 1
    red = sigmoid(x, gain, -1*offset_x)
    blue = 1-sigmoid(x, gain, offset_x)
    green = sigmoid(x, gain, offset_green) + (1-sigmoid(x,gain,-1*offset_green))
    green = green - 1.0
    return [blue * 255,green * 255,red * 255]

def _get_color(value):
    t = (value + ABS_LATENT_MAX_VALUE) / (ABS_LATENT_MAX_VALUE * 2.0)
    color = color_bar_rgb(t)
    return color

def create_color_panel(latent_spaces):
    images = []
    for z in latent_spaces:
        p = np.zeros((PANEL_HEIGHT, PANEL_WIDTH, 3))
        color = _get_color(z)
        p += color[::-1]
        p = np.clip(p, 0, 255).astype(np.uint8)  # Cast to uint8
        images.append(p)
    panel = np.concatenate(images, axis=1)
    return panel

#Create GUI

In [ ]:
image = widgets.Image(format='jpeg', width=320, height=240)
resize = widgets.Image(format='jpeg', width=160, height=80)
result = widgets.Image(format='jpeg', width=160, height=80)
diff_image = widgets.Image(format='jpeg', width=160, height=80)  # Error map
sample_image = widgets.Image(format='jpeg', width=160, height=80)  # Random sample
camera_link = traitlets.dlink((camera,'value'), (image,'value'), transform=bgr8_to_jpeg)
color_bar = widgets.Image(format='jpeg', width=VARIANTS_SIZE*PANEL_WIDTH, height=10*PANEL_HEIGHT)

# Loss component displays
rec_loss_widget = widgets.FloatText(description='REC Loss:')
kld_loss_widget = widgets.FloatText(description='KLD:')
perc_loss_widget = widgets.FloatText(description='PERC Loss:')
total_loss_widget = widgets.FloatText(description='Total Loss:')

display(image)
display(widgets.HBox([resize, result, diff_image]))
display(color_bar)
display(widgets.HBox([rec_loss_widget, kld_loss_widget, perc_loss_widget, total_loss_widget]))
display(sample_image)

## Start main process

Updated to use full forward pass, adjust for output range (assuming Sigmoid [0,1]; uncomment for Tanh [-1,1]), print latent stats, compute losses matching training, error map, random samples.

In [ ]:
def perceptual_loss(recon_x, x):
    # Adjust to [0,1] if needed (match training)
    # If VAE uses Tanh: x = (x + 1) / 2; recon_x = (recon_x + 1) / 2
    x_norm = vgg_normalization(x)
    recon_x_norm = vgg_normalization(recon_x)
    feat_recon = vgg(recon_x_norm)
    feat_x = vgg(x_norm)
    return F.mse_loss(feat_recon, feat_x, reduction='mean')

def vae_process(change):
    image_np = change['new']
    image = preprocess(image_np)
    # For resize: tensor (RGB) to np RGB
    resize_np = np.transpose(image.numpy() * 255, [1, 2, 0]).astype(np.uint8)
    resize.value = np_to_jpeg(resize_np)
    
    image = torch.unsqueeze(image, dim=0).to(device)
    
    # Use full forward to get recon, mu, logvar
    reconst, mu, logvar = vae(image)
    z = vae.reparameterize(mu, logvar)  # Recompute z if needed
    
    # Adjust reconst for visualization (clamp [0,1]; uncomment if Tanh: reconst = (reconst + 1) / 2)
    reconst_vis = torch.clamp(reconst, 0, 1)
    
    # Visualize reconstruction: to np RGB
    to_visualize = torch.squeeze(reconst_vis).detach().cpu().numpy()
    to_visualize = np.transpose(np.uint8(to_visualize * 255), [1, 2, 0])
    result.value = np_to_jpeg(to_visualize)
    
    # Latent space color bar
    latent_space = z.detach().cpu().numpy()[0]
    color_bar.value = np_to_jpeg(create_color_panel(latent_space))
    
    # Print latent stats
    print("Mu: mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}".format(
        mu.mean().item(), mu.std().item(), mu.min().item(), mu.max().item()))
    print("Logvar: mean={:.4f}, std={:.4f}, min={:.4f}, max={:.4f}".format(
        logvar.mean().item(), logvar.std().item(), logvar.min().item(), logvar.max().item()))
    print("Z: mean={:.4f}, std={:.4f}".format(z.mean().item(), z.std().item()))
    
    # Compute losses (BCE on [0,1])
    REC = F.binary_cross_entropy(reconst_vis, image, reduction='sum').item()
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()).item()
    p_loss = perceptual_loss(reconst_vis, image)
    PERC = p_loss.item() * reconst.numel()
    total_loss = REC + 4.0 * KLD + 0.05 * PERC
    
    # Update widgets
    rec_loss_widget.value = REC
    kld_loss_widget.value = KLD
    perc_loss_widget.value = PERC
    total_loss_widget.value = total_loss
    
    # Error map
    diff = torch.abs(reconst_vis - image).mean(dim=1, keepdim=True)
    diff = diff / diff.max() * 255
    diff_np = torch.squeeze(diff).detach().cpu().numpy()
    diff_np = cv2.cvtColor(np.uint8(diff_np), cv2.COLOR_GRAY2RGB)  # To RGB for consistency
    diff_image.value = np_to_jpeg(diff_np)
    
    # Random sample (10% chance)
    if np.random.rand() < 0.1:
        sample_z = torch.randn(1, VARIANTS_SIZE).to(device)
        sample_reconst = vae.decode(sample_z)
        sample_vis = torch.clamp(sample_reconst, 0, 1)  # Uncomment if Tanh: sample_vis = (sample_reconst + 1) / 2
        sample_vis = torch.squeeze(sample_vis).detach().cpu().numpy()
        sample_vis = np.transpose(np.uint8(sample_vis * 255), [1, 2, 0])
        sample_image.value = np_to_jpeg(sample_vis)

vae_process({'new': camera.value})
camera.observe(vae_process, names='value')

## Cleanup process

In [ ]:
camera.unobserve(vae_process, names='value')
camera.stop()
camera_link.unlink()